<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 ¿Por Qué Me Dijo que No? El Detective de las Decisiones 🕵️
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 08
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/08%20-%20Temas%20Avanzados%20de%20ML/Para%20Dummies/02_Explicabilidad_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 02 del Módulo 08 (Explicabilidad de modelos)](../02_Explicabilidad_de_Modelos.ipynb). Un modelo te dice «**préstamo negado**». Lo natural es preguntar: **¿por qué?** Aquí aprendemos a hacer de **detectives**: a descubrir **qué pistas pesaron más** en una decisión, y también **cuándo la explicación nos puede engañar**. Solo usamos *NumPy*, *scikit-learn* y gráficas.

Al terminar podrás explicar:
1. Cómo se **reparte el mérito** entre varios (la banda de tres amigos): la idea de los **valores de Shapley**.
2. Cómo saber qué pista **le importa al modelo** tapándola (barajando) y viendo cuánto se equivoca.
3. Por qué **dos pistas repetidas** confunden el reparto.
4. Por qué que el modelo **use** una pista **no significa** que **cause** el resultado (los helados y los ahogamientos).

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../02_Explicabilidad_de_Modelos.ipynb). Antes de esto, [cómo reutilizar lo aprendido](01_Transfer_Learning_Dummies.ipynb).

---
## 1. La banda de tres amigos y el reparto justo del mérito 🎸🥁🎤

Tres amigos tocan juntos: **guitarra (G)**, **batería (B)** y **voz (V)**. Cuando se presentan en un bar, el público paga según **quiénes tocan**:

| Quiénes tocan | Aplausos (propinas) |
|---|---|
| Nadie | 0 |
| Solo G | 10 |
| Solo B | 10 |
| Solo V | 20 |
| G + B | 30 (¡suenan mejor juntos!) |
| G + V | 40 |
| B + V | 50 |
| Los tres | **90** |

Al final hay **90** de propinas. **¿Cuánto le corresponde a cada uno?** El reparto **justo** (propuesto por el matemático Lloyd Shapley, 1953) es este: imagina **todos los órdenes** en los que los amigos podrían **subir al escenario**. Cada vez que alguien sube, **suma lo que aporta** a los que ya estaban. **Promedia** esos aportes **sobre todos los órdenes**. Eso es su **valor de Shapley**. Es exactamente lo que usan hoy muchos métodos para explicar modelos (cada «amigo» es una **pista** y las propinas son la **predicción**).

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

amigos = ["Guitarra", "Batería", "Voz"]
propinas = {(): 0, ("Guitarra",): 10, ("Batería",): 10, ("Voz",): 20,
            ("Guitarra", "Batería"): 30, ("Guitarra", "Voz"): 40, ("Batería", "Voz"): 50,
            ("Guitarra", "Batería", "Voz"): 90}

def ganancia(grupo):
    """Propinas de un grupo (sin importar el orden en que se nombren)."""
    return propinas[tuple(sorted(grupo, key=amigos.index))]

aporte_total = {a: 0.0 for a in amigos}
ordenes = list(itertools.permutations(amigos))
for orden in ordenes:                                         # cada orden posible de llegada al escenario
    presentes = []
    for a in orden:
        aporte_total[a] += ganancia(presentes + [a]) - ganancia(presentes)    # lo que SUMA al llegar
        presentes.append(a)
shapley = {a: aporte_total[a] / len(ordenes) for a in amigos}  # promedio sobre los 6 ordenes

print(f"Hay {len(ordenes)} órdenes posibles. Un ejemplo, el orden {ordenes[0]}: cada uno suma lo que aporta al llegar.")
for a in amigos:
    print(f"   {a:<9}: le corresponden {shapley[a]:.1f} de las 90 propinas")
print(f"Suma de los repartos = {sum(shapley.values()):.1f} (¡justo las 90 propinas! Nada se pierde ni se inventa)")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(amigos, [shapley[a] for a in amigos], color=["#2563eb", "#f59e0b", "#16a34a"])
ax.set_ylabel("Propinas que le corresponden"); ax.set_title("Reparto justo (valores de Shapley)", fontweight="bold")
for i, a in enumerate(amigos):
    ax.text(i, shapley[a] + 0.8, f"{shapley[a]:.1f}", ha="center")
plt.tight_layout(); plt.show()
assert abs(sum(shapley.values()) - 90) < 1e-9                   # las propinas se reparten completas
assert shapley["Voz"] > shapley["Batería"] > shapley["Guitarra"]

---
---
**Lección:** el reparto **suma exactamente 90** (la «propiedad de eficiencia»): ni sobra ni falta. La **voz** se lleva más porque **aporta más** a casi cualquier combinación. En un modelo, cada **pista** (ingresos, deudas, edad…) recibe así una **parte de la decisión**, y todas **suman la predicción**. Hacerlo con muchas pistas es costoso (los órdenes crecen muy rápido), por eso existen **atajos** que lo aproximan.

## 2. Taparle una pista al modelo: ¿cuánto le importa? 🙈

Otra forma de investigar es **tapar una pista y ver cuánto se equivoca** el modelo. Si **barajamos** (mezclamos al azar) los valores de una pista, esa pista **deja de decir algo útil**. Si el modelo **se equivoca mucho más**, esa pista **importaba**. Probamos con un modelo de **préstamos** con tres pistas: **ingresos**, **deudas** y **color favorito** (irrelevante a propósito). Luego añadimos una **copia exacta** de los ingresos para ver qué pasa con dos testigos que dicen lo mismo.

In [ ]:
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(1)
n = 3000
ingresos = rng.normal(0, 1, n)
deudas = rng.normal(0, 1, n)
color = rng.normal(0, 1, n)                                        # NO tiene nada que ver con el préstamo
aprobado = ((1.5 * ingresos - 1.0 * deudas + rng.normal(0, 0.7, n)) > 0).astype(int)

def importancia_barajando(modelo, X, y, nombres, repeticiones=20):
    """Cuanto BAJA el acierto al barajar cada pista (promedio de varias barajadas)."""
    g = np.random.default_rng(0)
    base = modelo.score(X, y)
    caidas = {}
    for j, nombre in enumerate(nombres):
        c = []
        for _ in range(repeticiones):
            Xb = X.copy(); Xb[:, j] = g.permutation(Xb[:, j])
            c.append(base - modelo.score(Xb, y))
        caidas[nombre] = np.mean(c)
    return base, caidas

X = np.column_stack([ingresos, deudas, color])
nombres = ["ingresos", "deudas", "color favorito"]
modelo = LogisticRegression().fit(X[:2000], aprobado[:2000])
base, imp = importancia_barajando(modelo, X[2000:], aprobado[2000:], nombres)

# Ahora, con una COPIA EXACTA de los ingresos (dos testigos que dicen lo mismo)
X2 = np.column_stack([ingresos, deudas, color, ingresos])
modelo2 = LogisticRegression().fit(X2[:2000], aprobado[:2000])
base2, imp2 = importancia_barajando(modelo2, X2[2000:], aprobado[2000:], nombres + ["ingresos (copia)"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(list(imp), list(imp.values()), color=["#2563eb", "#dc2626", "#94a3b8"]); axes[0].set_xlabel("Cuánto baja el acierto al taparla")
axes[0].set_title(f"Tres pistas (acierto normal: {base:.0%})", fontweight="bold", fontsize=10)
axes[1].barh(list(imp2), list(imp2.values()), color=["#2563eb", "#dc2626", "#94a3b8", "#60a5fa"]); axes[1].set_xlabel("Cuánto baja el acierto al taparla")
axes[1].set_title("Con una COPIA de los ingresos: ¡el mérito se reparte!", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
print("Importancia de los ingresos: sola =", round(imp["ingresos"], 3), "| con una copia: original =", round(imp2["ingresos"], 3), "y copia =", round(imp2["ingresos (copia)"], 3))
assert imp["color favorito"] < 0.01 and imp["ingresos"] > imp["deudas"] > imp["color favorito"]
assert imp2["ingresos"] < imp["ingresos"] and imp2["ingresos (copia)"] < imp["ingresos"]

---
---
**Lección:** tapar **ingresos** y **deudas** hace que el modelo falle mucho; tapar el **color favorito** casi no cambia nada: **no le importaba**. Pero cuando hay **dos pistas idénticas**, el modelo **reparte** su confianza entre ellas y **cada una parece menos importante** de lo que es. Como dos testigos que dicen lo mismo: si tapas a uno, el otro sigue hablando. **Conclusión:** una explicación **depende del modelo** y de **cómo están repetidas las pistas**.

## 3. Los helados y los ahogamientos: usar una pista no es causarla 🍦🌊

En verano **aumentan las ventas de helados** y también **los ahogamientos**. Un modelo que predice ahogamientos puede **apoyarse en los helados** (¡es una pista excelente!), y la explicación dirá «**los helados son lo más importante**». Pero **prohibir los helados no salvaría a nadie**: ambas cosas ocurren por el **calor** (la verdadera causa). Lo simulamos: hacemos un **mundo** donde el calor causa las dos cosas, entrenamos un modelo con los **helados**, y luego **«prohibimos» los helados** para ver qué pasa en el mundo real y qué predice el modelo.

In [ ]:
rng = np.random.default_rng(2)
dias = 2000
calor = rng.normal(0, 1, dias)                                             # la verdadera causa
helados = calor + rng.normal(0, 0.3, dias)                                 # el calor causa más helados (pista MUY limpia)
ahogados = np.maximum(0, 5 + 3 * calor + rng.normal(0, 1.0, dias))         # el calor causa más ahogamientos (pista MENOS limpia: solo helados y calor)

coef = np.polyfit(helados, ahogados, 1)                                    # el modelo aprende una recta: ahogados ≈ a·helados + b
predice = lambda h: np.maximum(0, np.polyval(coef, h))                       # (no puede predecir ahogamientos negativos)

# Explicación 1: ¿qué tan importante es la pista 'helados'? (barajar)
error = lambda h: np.mean(np.abs(predice(h) - ahogados))
bajo = error(helados); alto = np.mean([error(rng.permutation(helados)) for _ in range(20)])
print(f"Error del modelo con los helados reales: {bajo:.2f} ahogados | si barajamos los helados: {alto:.2f}  → «los helados son MUY importantes para el modelo».")

# Intervención: prohibir los helados (poner las ventas en un valor muy bajo). ¿Qué cambia en el MUNDO REAL?
helados_prohibidos = np.full(dias, helados.min())
ahogados_reales_tras_prohibir = ahogados                                    # el calor no cambió: los ahogamientos reales tampoco
prediccion_modelo = predice(helados_prohibidos)
print(f"Tras prohibir los helados: el MODELO predice {prediccion_modelo.mean():.1f} ahogamientos por día; en el MUNDO REAL siguen ocurriendo {ahogados_reales_tras_prohibir.mean():.1f}.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(helados, ahogados, s=6, color="#2563eb", alpha=0.5); axes[0].plot(np.sort(helados), predice(np.sort(helados)), color="#dc2626", lw=2, label="Lo que aprendió el modelo")
axes[0].set_xlabel("Ventas de helados"); axes[0].set_ylabel("Ahogamientos"); axes[0].legend(fontsize=8); axes[0].set_title("Los helados predicen muy bien los ahogamientos", fontweight="bold", fontsize=10)
axes[1].bar(["Mundo real\ntras prohibir", "Predicción del modelo\ntras prohibir"], [ahogados_reales_tras_prohibir.mean(), prediccion_modelo.mean()], color=["#16a34a", "#dc2626"])
axes[1].set_ylabel("Ahogamientos por día"); axes[1].set_title("El modelo se equivoca al «explicar» la causa", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert alto > 2 * bajo and prediccion_modelo.mean() < 0.6 * ahogados_reales_tras_prohibir.mean()

---
---
**Lección:** el modelo **usa mucho** los helados (explicación correcta de **cómo piensa el modelo**), pero **prohibirlos no cambia los ahogamientos** (**el modelo dice que bajarían**, y se equivoca). **Una explicación describe al modelo, no al mundo.** Por eso conviene decir «**el modelo se apoya en…**» y **no** «**esto causa…**». Y si un modelo decide sobre personas, **explicar** no basta: hay que **revisar sus resultados por grupos** (un modelo puede no mirar el género y aun así tratar distinto a hombres y mujeres usando otra pista que se le parece).

## Resumen en una frase 🎯

> **Explicar un modelo es como investigar un caso: se reparte el mérito entre las pistas (Shapley), se prueba qué pasa si se tapa una pista, se desconfía cuando hay pistas repetidas, y siempre se recuerda que «el modelo usa esta pista» no es lo mismo que «esta pista causa el resultado».**

### Lo que aprendiste hoy:
- ✅ **Shapley:** el reparto justo del mérito promedia el aporte de cada pista sobre **todos los órdenes**; **suma exactamente** la predicción.
- ✅ **Tapar una pista** (barajar) mide **cuánto le importa al modelo**: el color favorito no importaba, los ingresos sí.
- ✅ **Pistas repetidas** (dos testigos que dicen lo mismo) **reparten** el mérito y confunden la lectura.
- ✅ **Usar una pista ≠ causar** (helados y ahogamientos): una explicación describe **al modelo**, no al mundo; en decisiones sobre personas hay que **auditar resultados por grupos**.

> 🎓 **Siguiente paso:** el [cuaderno estándar](../02_Explicabilidad_de_Modelos.ipynb) implementa Shapley **exacto** (con sus axiomas verificados), PDP, LIME y *gradientes integrados*; y luego, [la aguja en el pajar (clases desbalanceadas)](03_Desbalanceados_Dummies.ipynb).

---
##### 🛠️ Práctica 1: Un amigo que no aporta nada: el reparto de Shapley le da cero

Cambiemos la banda: ahora la **guitarra no suma nada** (con ella o sin ella las propinas son iguales). Usa `amigos`, `ordenes` y el mismo método de la Sección 1 (sumar lo que aporta cada uno al llegar, promediando sobre los 6 órdenes):

1. Crea la tabla `propinas2`: nadie `0`; solo G `0`; solo B `20`; solo V `30`; G+B `20`; G+V `30`; B+V `70`; los tres `70`.
2. Define `ganancia2(grupo)` igual que `ganancia`, pero leyendo `propinas2`.
3. Recorre cada orden de `ordenes`, acumula el aporte de cada amigo en `aporte2` y calcula `shapley2` (el aporte total de cada uno dividido entre 6).
4. Comprueba con `assert` que la guitarra recibe `0`, que el reparto suma `70` y que la voz recibe más que la batería.

In [ ]:
# Escribe tu código aquí

# propinas2 = {(): 0, ("Guitarra",): 0, ("Batería",): 20, ("Voz",): 30,
#              ("Guitarra", "Batería"): ..., ("Guitarra", "Voz"): ..., ("Batería", "Voz"): ...,
#              ("Guitarra", "Batería", "Voz"): ...}
# def ganancia2(grupo):
#     return propinas2[tuple(sorted(grupo, key=amigos.index))]
#
# aporte2 = {a: 0.0 for a in amigos}
# for orden in ordenes:
#     presentes = []
#     for a in orden:
#         aporte2[a] += ganancia2(presentes + [a]) - ganancia2(presentes)
#         presentes.append(a)
# shapley2 = {a: aporte2[a] / ... for a in amigos}


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
propinas2 = {(): 0, ("Guitarra",): 0, ("Batería",): 20, ("Voz",): 30,
             ("Guitarra", "Batería"): 20, ("Guitarra", "Voz"): 30, ("Batería", "Voz"): 70,
             ("Guitarra", "Batería", "Voz"): 70}

def ganancia2(grupo):
    return propinas2[tuple(sorted(grupo, key=amigos.index))]

aporte2 = {a: 0.0 for a in amigos}
for orden in ordenes:                                         # los 6 ordenes de llegada
    presentes = []
    for a in orden:
        aporte2[a] += ganancia2(presentes + [a]) - ganancia2(presentes)    # lo que SUMA al llegar
        presentes.append(a)
shapley2 = {a: aporte2[a] / len(ordenes) for a in amigos}

for a in amigos:
    print(f"{a:<9}: le corresponden {shapley2[a]:.1f} de las 70 propinas")

assert abs(shapley2["Guitarra"]) < 1e-9                       # quien no aporta nada, no recibe nada
assert abs(sum(shapley2.values()) - 70) < 1e-9                # y las propinas se reparten completas
assert shapley2["Voz"] > shapley2["Batería"]
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. En la banda de tres amigos, ¿por qué el reparto de Shapley suma exactamente 90? ¿Qué pasaría si repartiéramos solo mirando cuánto aporta cada uno **al llegar primero**?
2. Un banco te dice que tu préstamo fue negado porque «tus ingresos pesaron más». ¿Qué **sí** y qué **no** puedes concluir? ¿Qué le preguntarías?
3. Si dos pistas dicen exactamente lo mismo, ¿por qué cada una parece «menos importante» de lo que realmente es la información que comparten? ¿Cómo podrías comunicar la importancia **del conjunto**?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>